# 🎵 ACE-Step 1.5 — GPU Music Generator

**Előfeltétel**: `Runtime → Change runtime type → T4 GPU`

**Pipeline**:
1. `ace-qwen3` — dalszöveg + audio kód generálás (LM)
2. `dit-vae` — audio szintézis WAV/MP3-ba (DiT + VAE)

Modellek Google Drive-on cache-elve → második futtatástól nem kell újra letölteni.

In [ ]:
# ── 1. Környezet diagnosztika ─────────────────────────────────────────────────
import subprocess, os, shutil

def chk(label, ok, detail=''):
    icon = '✅' if ok else '❌'
    print(f'  {icon}  {label}' + (f'  →  {detail}' if detail else ''))
    return ok

print('=== GPU ===')
smi = subprocess.run(['nvidia-smi', '--query-gpu=name,memory.total',
                      '--format=csv,noheader'], capture_output=True, text=True)
gpu_ok = chk('nvidia-smi', smi.returncode == 0, smi.stdout.strip() or smi.stderr.strip())
if not gpu_ok:
    raise SystemExit('❌ Nincs GPU!  Runtime → Change runtime type → T4 GPU')

print('\n=== CUDA toolkit ===')
nvcc = shutil.which('nvcc') or '/usr/local/cuda/bin/nvcc'
chk('nvcc', os.path.exists(nvcc), nvcc)
cuda_ver = subprocess.run([nvcc, '--version'], capture_output=True, text=True)
if cuda_ver.returncode == 0:
    ver_line = [l for l in cuda_ver.stdout.splitlines() if 'release' in l]
    chk('CUDA verzió', True, ver_line[0].strip() if ver_line else '?')

print('\n=== Build eszközök ===')
for tool in ['cmake', 'make', 'gcc', 'g++', 'git']:
    path = shutil.which(tool)
    chk(tool, bool(path), path or 'HIÁNYZIK')

print('\n=== Python csomagok ===')
for pkg in ['huggingface_hub']:
    try:
        import importlib; m = importlib.import_module(pkg.replace('-','_'))
        chk(pkg, True, getattr(m, '__version__', 'ok'))
    except ImportError:
        chk(pkg, False, 'pip install szükséges')

print('\nDiagnosztika kész.')

In [ ]:
# ── 2. Google Drive csatolása (modellek cache-eléséhez) ───────────────────────
from google.colab import drive
drive.mount('/content/drive')

DRIVE_MODELS = '/content/drive/MyDrive/ACE-Step-models'
os.makedirs(DRIVE_MODELS, exist_ok=True)
print(f'✅ Drive modell mappa: {DRIVE_MODELS}')

In [ ]:
# ── 3. Repo klónozás + CUDA build  (Drive cache-el — egyszer kell csak) ───────
import time, shutil, os, subprocess, tarfile

REPO    = '/content/acestep.cpp'
BUILD   = f'{REPO}/build'
ACE_BIN = f'{BUILD}/ace-qwen3'
DIT_BIN = f'{BUILD}/dit-vae'

# Drive cache elérési útja (Drive csatolása a 2. cellában történt)
DRIVE_BUILD_TAR = '/content/drive/MyDrive/ACE-Step-models/acestep-build-cuda75.tar.gz'

# ── 3a. Swap fájl (OOM védelem build közben) ──────────────────────────────────
if not os.path.exists('/swapfile'):
    !fallocate -l 8G /swapfile && chmod 600 /swapfile && mkswap /swapfile && swapon /swapfile
!free -h | grep -E "Mem|Swap"

# ── 3b. Build visszatöltése Drive-ról (ha már egyszer megcsináltuk) ───────────
if not (os.path.exists(ACE_BIN) and os.path.exists(DIT_BIN)):
    if os.path.exists(DRIVE_BUILD_TAR):
        print('📦 Build visszatöltése Drive-ról (~30 mp)...')
        t = time.time()
        # Repo kell a shared lib-ekhez és headerekhez
        if not os.path.exists(REPO):
            !git clone --depth=1 https://github.com/ServeurpersoCom/acestep.cpp {REPO}
            !cd {REPO} && git submodule update --init --depth=1
        os.makedirs(BUILD, exist_ok=True)
        with tarfile.open(DRIVE_BUILD_TAR, 'r:gz') as tar:
            tar.extractall('/content/acestep.cpp')
        print(f'✅ Build visszatöltve Drive-ról  ({time.time()-t:.0f} mp)')
    else:
        # ── Első alkalom: klón + build + mentés Drive-ra ──────────────────────
        if not os.path.exists(REPO):
            t = time.time()
            print('Klónozás (shallow)...')
            !git clone --depth=1 https://github.com/ServeurpersoCom/acestep.cpp {REPO}
            !cd {REPO} && git submodule update --init --depth=1
            print(f'✅ Klónozva  ({time.time()-t:.0f} mp)')
        os.makedirs(BUILD, exist_ok=True)

        # nvcc
        cuda_root = '/usr/local/cuda'
        nvcc_path = shutil.which('nvcc') or f'{cuda_root}/bin/nvcc'
        if not os.path.exists(nvcc_path):
            raise RuntimeError('nvcc nem található!  Runtime → Change runtime type → T4 GPU')
        print(f'nvcc: {nvcc_path}')
        !{nvcc_path} --version 2>&1 | grep release

        print('\n── cmake configure ──')
        !cd {BUILD} && cmake .. \
            -DGGML_CUDA=ON \
            -DCMAKE_BUILD_TYPE=Release \
            -DCMAKE_CUDA_COMPILER={nvcc_path} \
            -DCMAKE_CUDA_ARCHITECTURES=75 \
            -DGGML_CUDA_FA_ALL_QUANTS=OFF \
            2>&1 | tail -15

        # -DGGML_CUDA_FA_ALL_QUANTS=OFF : kihagyja a Flash Attention template
        # variansokt (fattn-tile-instance-*.cu) — ezek a leglassabbak,
        # ACE-Step-hez nem szüksegesek, build: ~3 perc helyett ~30 perc
        print('\n── cmake build -j2  (~3-5 perc) ──')
        t = time.time()
        !cd {BUILD} && cmake --build . --config Release -j2 2>&1
        elapsed = time.time() - t

        if os.path.exists(ACE_BIN) and os.path.exists(DIT_BIN):
            print(f'\n✅ Build kész  ({elapsed:.0f} mp)')
            # ── Mentés Drive-ra → következő session azonnali ──────────────────
            print('💾 Build mentése Drive-ra (következő sessionhöz)...')
            t2 = time.time()
            with tarfile.open(DRIVE_BUILD_TAR, 'w:gz') as tar:
                tar.add(BUILD, arcname='build')
            sz = os.path.getsize(DRIVE_BUILD_TAR) / 1024**2
            print(f'✅ Drive-ra mentve: {DRIVE_BUILD_TAR}  ({sz:.0f} MB, {time.time()-t2:.0f} mp)')
            print('   → Következő session: build kihagyva, ~30 mp alatt betölt!')
        else:
            print('❌ Build SIKERTELEN — nézd a fenti hibát')
else:
    print('✅ Binárisok már a memóriában vannak')

# ── Végeredmény ───────────────────────────────────────────────────────────────
print()
for p in [ACE_BIN, DIT_BIN]:
    icon = '✅' if os.path.exists(p) else '❌'
    print(f'  {icon}  {p}')

In [ ]:
# ── 4. Modellek letöltése (Drive cache) ───────────────────────────────────────
# Profil választó:
#   'q8'  = LM Q8_0  + DiT Q8_0   ~7.7 GB  ← AJÁNLOTT  (legjobb minőség, T4-en elfér)
#   'q5x' = LM Q5_K_M + DiT Q8_0  ~6.5 GB  ← jó kompromisszum
#   'q5'  = LM Q5_K_M + DiT Q5_K_M ~5.5 GB  ← kisebb VRAM
# ⚠️  NE használj DiT Q4_K_M-et — túl agresszív kvantálás, fehérzajt generál!

PROFILE = 'q8'  # @param ['q8', 'q5x', 'q5']

HF_REPO = 'Serveurperso/ACE-Step-1.5-GGUF'
MODELS_DIR = f'{REPO}/models'
os.makedirs(MODELS_DIR, exist_ok=True)

MODEL_FILES = {
    'q8': [
        'acestep-5Hz-lm-4B-Q8_0.gguf',
        'Qwen3-Embedding-0.6B-Q8_0.gguf',
        'acestep-v15-turbo-Q8_0.gguf',
        'vae-BF16.gguf',
    ],
    'q5x': [
        'acestep-5Hz-lm-4B-Q5_K_M.gguf',
        'Qwen3-Embedding-0.6B-Q8_0.gguf',
        'acestep-v15-turbo-Q8_0.gguf',   # DiT marad Q8!
        'vae-BF16.gguf',
    ],
    'q5': [
        'acestep-5Hz-lm-4B-Q5_K_M.gguf',
        'Qwen3-Embedding-0.6B-Q8_0.gguf',
        'acestep-v15-turbo-Q5_K_M.gguf',
        'vae-BF16.gguf',
    ],
}

!pip install -q huggingface_hub
from huggingface_hub import hf_hub_download

for fname in MODEL_FILES[PROFILE]:
    local_path  = f'{MODELS_DIR}/{fname}'
    drive_path  = f'{DRIVE_MODELS}/{fname}'

    if os.path.exists(local_path):
        print(f'  [SKIP local] {fname}')
    elif os.path.exists(drive_path):
        print(f'  [Drive→local] {fname}')
        os.symlink(drive_path, local_path)   # szimlink: nem másol, gyors
    else:
        print(f'  [HF letöltés] {fname} ...')
        hf_hub_download(repo_id=HF_REPO, filename=fname, local_dir=MODELS_DIR)
        # Mentés Drive-ra a következő futtatáshoz
        import shutil
        shutil.copy2(local_path, drive_path)
        print(f'  [Drive mentve] {fname}')

# Modell névleképzés a profilhoz
LM_NAMES = {
    'q8':  'acestep-5Hz-lm-4B-Q8_0.gguf',
    'q5x': 'acestep-5Hz-lm-4B-Q5_K_M.gguf',
    'q5':  'acestep-5Hz-lm-4B-Q5_K_M.gguf',
}
DIT_NAMES = {
    'q8':  'acestep-v15-turbo-Q8_0.gguf',
    'q5x': 'acestep-v15-turbo-Q8_0.gguf',   # DiT marad Q8 a q5x profilban
    'q5':  'acestep-v15-turbo-Q5_K_M.gguf',
}
LM_MODEL     = f'{MODELS_DIR}/{LM_NAMES[PROFILE]}'
TEXT_ENCODER = f'{MODELS_DIR}/Qwen3-Embedding-0.6B-Q8_0.gguf'
DIT_MODEL    = f'{MODELS_DIR}/{DIT_NAMES[PROFILE]}'
VAE_MODEL    = f'{MODELS_DIR}/vae-BF16.gguf'

print('\n✅ Modellek készen')

In [ ]:
# ── 5. Dal generáló függvény ───────────────────────────────────────────────────
import json, time, subprocess, glob
from pathlib import Path
from IPython.display import Audio, display, HTML

OUTPUT_DIR = '/content/output'
os.makedirs(OUTPUT_DIR, exist_ok=True)

def generate_song(caption: str, lyrics: str,
                  lang: str = 'en',
                  steps: int = 8,
                  lm_batch: int = 1,
                  dit_batch: int = 1,
                  seed: int = -1,
                  bpm: int = 0,
                  duration: float = 0,
                  keyscale: str = '',
                  timesignature: str = ''):
    """
    caption       : stílus leírás angolul
    lyrics        : dalszöveg [verse]/[chorus] tagekkel
    lang          : vocal_language (en/zh/ja/ko/fr/de/es/pt/ru/it)
    steps         : inference_steps (turbo=8, sft=32-50)
    lm_batch      : hány különböző audio kód variáció (különböző dal)
    dit_batch     : hány render variáció (finom különbségek)
    bpm/duration/keyscale/timesignature : ha megadod, a modell kihagyja
                    a Phase 1 (metadata CoT) lépést és azonnal audio
                    kódokat generál — gyorsabb és megbízhatóbb!
    """
    ts = int(time.time())
    req_file = f'/tmp/request_{ts}.json'

    req = {
        'caption': caption,
        'lyrics': lyrics,
        'inference_steps': steps,
        'guidance_scale': 1.0,
        'lm_cfg_scale':   3.0,   # magasabb = erősebb vokál kondicionálás (default: 2.0)
        'lm_negative_prompt': 'instrumental, no vocals, no singing, background music only',
        'shift': 3.0,
        'vocal_language': lang,
    }
    # Ha minden metadata meg van adva → Phase 1 CoT kihagyva, közvetlen kódgenerálás
    if bpm:           req['bpm'] = bpm
    if duration:      req['duration'] = duration
    if keyscale:      req['keyscale'] = keyscale
    if timesignature: req['timesignature'] = timesignature
    if seed != -1:    req['seed'] = seed

    all_meta = bpm and duration and keyscale and timesignature
    print(f'  Mód: {"⚡ közvetlen (Phase1 kihagyva)" if all_meta else "🔄 CoT metadata fill (Phase1 fut)"}')

    with open(req_file, 'w') as f:
        json.dump(req, f, ensure_ascii=False, indent=2)

    # 1. lépés: ace-qwen3
    print('🎼 1/2  Audio kódok generálása (ace-qwen3)...')
    t1 = time.time()
    cmd1 = [ACE_BIN, '--request', req_file, '--model', LM_MODEL]
    if lm_batch > 1: cmd1 += ['--batch', str(lm_batch)]
    r1 = subprocess.run(cmd1, capture_output=True, text=True)
    if r1.returncode != 0:
        print('❌ ace-qwen3 hiba:\n', r1.stderr[-2000:])
        return
    print(f'   ✅ {time.time()-t1:.0f} mp')

    # CoT összefoglaló
    combined_out = (r1.stdout or '') + (r1.stderr or '')
    for line in combined_out.splitlines():
        if any(k in line for k in ('bpm:', 'keyscale:', 'duration:', 'language:', '[Batch')):
            print('  ', line.strip())

    # 2. lépés: dit-vae (minden LM variációra)
    req_base = Path(req_file).stem
    req_dir  = Path(req_file).parent
    results  = []

    for i in range(lm_batch):
        gen_req = req_dir / f'{req_base}{i}.json'

        if not gen_req.exists():
            alt = Path('/content') / f'{req_base}{i}.json'
            if alt.exists():
                import shutil as _sh; _sh.copy2(str(alt), str(gen_req))
            else:
                print(f'  ❌ {gen_req} nem található')
                continue

        print(f'\n🎶 2/2  Audio szintézis (dit-vae, variáció {i})...')
        t2 = time.time()
        cmd2 = [
            DIT_BIN,
            '--request',      str(gen_req),
            '--text-encoder', TEXT_ENCODER,
            '--dit',          DIT_MODEL,
            '--vae',          VAE_MODEL,
        ]
        if dit_batch > 1: cmd2 += ['--batch', str(dit_batch)]

        # dit-vae a request JSON mellé ír → cwd = req_dir
        r2 = subprocess.run(cmd2, capture_output=True, text=True, cwd=str(req_dir))

        # ── dit-vae kimenet mindig látszik ───────────────────────────────────
        dit_out = (r2.stdout or '') + (r2.stderr or '')
        print('  dit-vae kimenet:')
        for line in dit_out.splitlines()[:30]:
            print('   ', line)

        if r2.returncode != 0:
            print(f'❌ dit-vae kilépési kód: {r2.returncode}')
            continue
        elapsed2 = time.time()-t2
        print(f'   ✅ {elapsed2:.0f} mp')

        # MP3/WAV fájlok áthelyezése req_dir-ből OUTPUT_DIR-be
        for ext in ('*.mp3', '*.wav'):
            for src in sorted(glob.glob(str(req_dir / ext))):
                dst = f'{OUTPUT_DIR}/{Path(src).name}'
                shutil.move(src, dst)
                results.append(dst)
                sz = os.path.getsize(dst) / 1024
                print(f'  📁 {Path(dst).name}  ({sz:.0f} KB)')
                display(Audio(dst, autoplay=False))

    print(f'\n✅ Kész! Fájlok: {OUTPUT_DIR}')
    return results

print('✅ generate_song() betöltve')

In [ ]:
# ── 6. GENERÁLÁS — itt módosítsd ──────────────────────────────────────────────
# ── Modellek: mindig Q8 (vokálhoz szükséges, Q5-nél nem generál éneket) ───────
import os
from pathlib import Path

_MDIR = '/content/acestep.cpp/models'
LM_MODEL     = f'{_MDIR}/acestep-5Hz-lm-4B-Q8_0.gguf'
TEXT_ENCODER = f'{_MDIR}/Qwen3-Embedding-0.6B-Q8_0.gguf'
DIT_MODEL    = f'{_MDIR}/acestep-v15-turbo-Q8_0.gguf'
VAE_MODEL    = f'{_MDIR}/vae-BF16.gguf'

_all_ok = True
for _p in [LM_MODEL, TEXT_ENCODER, DIT_MODEL, VAE_MODEL]:
    _ok = os.path.exists(_p)
    if not _ok: _all_ok = False
    print(f'  {"✅" if _ok else "❌"} {Path(_p).name}')
if not _all_ok:
    raise RuntimeError('❌ Hiányzó Q8 modellek — futtasd újra a 4. cellát PROFILE="q8"-cal!')

CAPTION = "Melancholic Hungarian folk ballad with acoustic guitar and violin, \
emotional male vocals, slow tempo, minor key, authentic Eastern European feel"

LYRICS = """[Verse]
Kimegyek a mezőre hajnalban
Harmat csillog a fűszálakon
Gondolok rád minden pillanatban
Elmúlt az idő és itt állok magam

[Chorus]
Hová tűnt a régi szép világ
Ahol minden álom megtalált
Most csak a szél súg a fülembe
Hogy nem leszel már soha az enyém

[Verse]
Felszállnak a darvak az égen
Messzire repülnek el tőlem
Magamra maradtam e csendben
Csak az emlék él már a szívemben

[Chorus]
Hová tűnt a régi szép világ
Ahol minden álom megtalált
Most csak a szél súg a fülembe
Hogy nem leszel már soha az enyém

[Outro]
Elmúlt az idő elmúlt minden
Csak te maradsz meg az álmaimban"""

results = generate_song(
    caption       = CAPTION,
    lyrics        = LYRICS,
    lang          = '',          # auto-detect → model érzékeli a magyart
    steps         = 8,           # turbo=8, sft=32-50
    lm_batch      = 1,           # hány különböző verzió
    dit_batch     = 1,
    # ── Metadata megadva → Phase 1 CoT kihagyva, közvetlen kódgenerálás ──
    bpm           = 76,
    duration      = 120,
    keyscale      = 'D minor',
    timesignature = '4',
)

In [ ]:
# ── 7. Generált dalok mentése Drive-ra ────────────────────────────────────────
import shutil, glob

DRIVE_OUTPUT = '/content/drive/MyDrive/ACE-Step-output'
os.makedirs(DRIVE_OUTPUT, exist_ok=True)

saved = 0
for f in glob.glob(f'{OUTPUT_DIR}/*.mp3') + glob.glob(f'{OUTPUT_DIR}/*.wav'):
    dest = f'{DRIVE_OUTPUT}/{Path(f).name}'
    shutil.copy2(f, dest)
    print(f'✅ Drive-ra mentve: {dest}')
    saved += 1

if saved == 0:
    print('Nincs menteni való fájl az output könyvtárban.')

In [ ]:
# ── 8. Build cache mentése Drive-ra (egyszer kell csak!) ──────────────────────
# Futtasd a build után → következő session 30 mp indulás 10 perc helyett
import tarfile, os, time

DRIVE_BUILD_TAR = '/content/drive/MyDrive/ACE-Step-models/acestep-build-cuda75.tar.gz'
BUILD = '/content/acestep.cpp/build'

if os.path.exists(f'{BUILD}/ace-qwen3') and os.path.exists(f'{BUILD}/dit-vae'):
    if os.path.exists(DRIVE_BUILD_TAR):
        print(f'⚠️  Drive cache már létezik: {DRIVE_BUILD_TAR}')
        print('    Felülíráshoz töröld kézzel a Drive-ról, majd futtasd újra.')
    else:
        print('💾 Build mentése Drive-ra...')
        t = time.time()
        with tarfile.open(DRIVE_BUILD_TAR, 'w:gz') as tar:
            tar.add(BUILD, arcname='build')
        sz = os.path.getsize(DRIVE_BUILD_TAR) / 1024**2
        print(f'✅ Kész!  {sz:.0f} MB  →  {DRIVE_BUILD_TAR}  ({time.time()-t:.0f} mp)')
        print()
        print('📋 Következő session — indulás ~2 perc:')
        print('   1. Cella 2: Drive csatolás')
        print('   2. Cella 3: build Drive-ból tölt be (~30 mp)')
        print('   3. Cella 4: modellek symlink Drive-ról (~5 mp)')
        print('   4. Cella 6: generálás GPU-n ~2-3 perc/dal  🚀')
else:
    print('❌ Binárisok nem találhatók — a build még nem kész vagy sikertelen volt.')